# Ablacao: Retweet Normalizado — HoTHP vs RoTHP

**Hipotese:** O desempenho ruim do HoTHP no dataset retweet se deve a escala temporal extrema
(mean_dt=2750, max_dt=582928), que causa overflow no kernel hiperbolico.

**Experimento:** Normalizar os timestamps do retweet dividindo pelo gap medio de cada sequencia
(mean_dt -> 1.0, max_dt -> ~227) e re-treinar ambos os modelos.

Se HoTHP melhorar significativamente apos normalizacao, confirma-se que o problema
e de escala numerica e nao de capacidade do modelo.

**Protocolo:**
- 3 seeds (2019, 2020, 2021), 100 epocas, mesmos hiperparametros
- Avaliacao em 1x (max_len=50, mesma condicao do treino) e 5x (max_len=250, extrapolacao)
- Comparacao com resultados do benchmark original (sem normalizacao)

**Run on Colab:** Runtime -> Change runtime type -> T4 GPU

In [ ]:
# ── Instalacao e setup ────────────────────────────────────────────────
import os, sys

# ── Clona o repositorio ufc-easytpp ──────────────────────────────────
if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git

sys.path.insert(0, 'ufc-easytpp')

# ── Instala dependencias ─────────────────────────────────────────────
!pip install omegaconf datasets pyyaml matplotlib pandas seaborn tqdm -q

# ── Fix: __init__.py minimo (evita imports de modelos que precisam de deps extras) ──
_init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
with open(_init_path, 'w') as f:
    f.write(
        "from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel\n"
        "from easy_tpp.model.torch_model.torch_thp    import THP    as TorchTHP\n"
        "from easy_tpp.model.torch_model.torch_rothp  import RoTHP  as TorchRoTHP\n"
        "from easy_tpp.model.torch_model.torch_hothp  import HoTHP  as TorchHoTHP\n"
    )

# ── Fix: forca import de todos os modelos no runner ──────────────────
_runner_path = 'ufc-easytpp/easy_tpp/runner/tpp_runner.py'
with open(_runner_path, 'r') as f:
    _content = f.read()
if 'import easy_tpp.model' not in _content:
    _content = _content.replace(
        'from collections import OrderedDict\n',
        'from collections import OrderedDict\nimport easy_tpp.model  # noqa: F401\n'
    )
    with open(_runner_path, 'w') as f:
        f.write(_content)

import torch
GPU = 0 if torch.cuda.is_available() else -1
print(f'OK — GPU={GPU}, torch={torch.__version__}', flush=True)

import os, sys, json, time, pickle, gc, re, io, contextlib, tempfile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml
%matplotlib inline

In [ ]:
# ── Informacoes de hardware ───────────────────────────────────────────
import platform, psutil, datetime

hw = {}

# GPU
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    hw['gpu_name']       = props.name
    hw['gpu_vram_gb']    = round(props.total_memory / 1024**3, 2)
    hw['gpu_sm_count']   = props.multi_processor_count
    hw['cuda_version']   = torch.version.cuda
else:
    hw['gpu_name']       = 'CPU only'
    hw['gpu_vram_gb']    = 0
    hw['gpu_sm_count']   = 0
    hw['cuda_version']   = 'N/A'

# CPU / RAM
hw['cpu']        = platform.processor() or platform.machine()
hw['cpu_cores']  = psutil.cpu_count(logical=False)
hw['cpu_threads']= psutil.cpu_count(logical=True)
hw['ram_gb']     = round(psutil.virtual_memory().total / 1024**3, 2)
hw['os']         = f'{platform.system()} {platform.release()}'
hw['torch']      = torch.__version__
hw['python']     = platform.python_version()
hw['timestamp']  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print('=' * 50, flush=True)
print('          HARDWARE DO AMBIENTE', flush=True)
print('=' * 50, flush=True)
print(f'  GPU        : {hw["gpu_name"]}', flush=True)
print(f'  VRAM       : {hw["gpu_vram_gb"]} GB', flush=True)
print(f'  CUDA       : {hw["cuda_version"]}', flush=True)
print(f'  CPU        : {hw["cpu"][:40]}', flush=True)
print(f'  Nucleos    : {hw["cpu_cores"]} fisicos / {hw["cpu_threads"]} threads', flush=True)
print(f'  RAM        : {hw["ram_gb"]} GB', flush=True)
print(f'  PyTorch    : {hw["torch"]}', flush=True)
print(f'  Python     : {hw["python"]}', flush=True)
print(f'  Inicio     : {hw["timestamp"]}', flush=True)
print('=' * 50, flush=True)

In [ ]:
# ── Gerar dataset retweet normalizado ─────────────────────────────────
# Download do HuggingFace e normalizacao: timestamps / mean_gap por sequencia
# Resultado: mean_dt ~= 1.0, max_dt ~= 227

from datasets import load_dataset
from pathlib import Path

OUTPUT_DIR = Path('./datasets/retweet_norm')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split in ['train', 'validation', 'test']:
    print(f"Processing {split}...", flush=True)
    ds = load_dataset('easytpp/retweet', split=split)
    sequences = []
    for row in ds:
        t = np.array(row['time_since_start'], dtype=np.float64)
        types = list(row['type_event'])
        dim = row.get('dim_process', 3)
        if len(t) < 3:
            continue
        gaps = np.diff(t)
        mean_gap = gaps[gaps > 0].mean() if np.any(gaps > 0) else 1.0
        if mean_gap <= 0:
            mean_gap = 1.0
        t_norm = (t - t[0]) / mean_gap
        dt_norm = np.diff(t_norm)
        dt_norm = np.insert(dt_norm, 0, 0.0)
        sequences.append({
            'time_since_start': t_norm.tolist(),
            'time_since_last_event': dt_norm.tolist(),
            'type_event': types,
            'dim_process': dim,
        })
    out_file = OUTPUT_DIR / f'{split}.json'
    with open(out_file, 'w') as f:
        json.dump(sequences, f)
    print(f"  {len(sequences)} seqs -> {out_file}", flush=True)

# Verificar estatisticas
with open(OUTPUT_DIR / 'train.json') as f:
    train_data = json.load(f)
all_dts = []
for seq in train_data[:500]:
    dts = np.diff(seq['time_since_start'])
    all_dts.extend(dts[dts > 0].tolist())
all_dts = np.array(all_dts)
print(f"\nStats (train, primeiras 500 seqs):", flush=True)
print(f"  mean_dt = {all_dts.mean():.4f}  (esperado ~= 1.0)", flush=True)
print(f"  max_dt  = {all_dts.max():.2f}", flush=True)
print(f"  CV      = {all_dts.std()/all_dts.mean():.3f}", flush=True)

In [ ]:
# ── Configuracao do experimento ───────────────────────────────────────

MODELS = ['RoTHP', 'HoTHP']
SEEDS = [2019, 2020, 2021]
NUM_EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 64
NUM_HEADS = 2
NUM_LAYERS = 2
DROPOUT = 0.1
TIME_EMB_SIZE = 16
TRAIN_MAX_LEN = 50
EVAL_1X = 50
EVAL_5X = 250
NUM_EVENT_TYPES = 3
DATASET_NAME = 'retweet_norm'

# Progresso persistente (sobrevive a quedas de conexao)
PROGRESS_FILE = 'retweet_norm_ablation_progress.pkl'

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'rb') as f:
            data = pickle.load(f)
        n_trained = len(data.get('trained', set()))
        n_results = len(data.get('results', []))
        print(f'Progresso carregado: {n_trained} treinados, {n_results} evals', flush=True)
        return data
    return {'results': [], 'trained': set(), 'epoch_logs': {}, 'train_times': {}}

def save_progress(prog):
    with open(PROGRESS_FILE, 'wb') as f:
        pickle.dump(prog, f)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'Modelos: {MODELS}', flush=True)
print(f'Seeds:   {SEEDS}', flush=True)
print(f'Epocas:  {NUM_EPOCHS}', flush=True)
print(f'Train max_len: {TRAIN_MAX_LEN}  |  Eval 1x: {EVAL_1X}  |  Eval 5x: {EVAL_5X}', flush=True)

In [ ]:
# ── Gerador de config YAML ────────────────────────────────────────────

from easy_tpp.config_factory import Config
from easy_tpp.runner import Runner

def make_yaml(model_id, seed, max_len, stage='train', pretrained_model_dir=None):
    """Gera config dict YAML para treino ou eval no retweet normalizado."""
    exp_id = f'{DATASET_NAME}_{model_id}_s{seed}_{stage}'

    model_cfg = {
        'hidden_size':    HIDDEN_SIZE,
        'num_heads':      NUM_HEADS,
        'num_layers':     NUM_LAYERS,
        'dropout':        DROPOUT,
        'time_emb_size':  TIME_EMB_SIZE,
        'use_ln':         False,
        'thinning': {
            'num_sample': 1, 'num_exp': 500, 'look_ahead_time': 10,
            'patience_counter': 5, 'over_sample_rate': 5,
            'num_samples_boundary': 5, 'dtime_max': 10,
            'num_seq': 10, 'num_step_gen': 1,
        },
    }

    if stage == 'train':
        model_cfg['loss_integral_num_sample_per_step'] = 20
        model_cfg['mc_num_sample_per_step'] = 20
    if pretrained_model_dir:
        model_cfg['pretrained_model_dir'] = pretrained_model_dir

    trainer_cfg = {
        'batch_size': BATCH_SIZE,
        'max_epoch':  NUM_EPOCHS if stage == 'train' else 1,
        'seed':       seed,
        'gpu':        GPU,
        'metrics':    ['acc', 'rmse'],
    }
    if stage == 'train':
        trainer_cfg.update({
            'valid_freq': 1, 'use_tfb': False,
            'optimizer': 'adam', 'learning_rate': LEARNING_RATE,
            'shuffle': False,
        })

    config = {
        'pipeline_config_id': 'runner_config',
        'data': {
            DATASET_NAME: {
                'data_format': 'json',
                'train_dir': './datasets/retweet_norm/train.json',
                'valid_dir': './datasets/retweet_norm/validation.json',
                'test_dir':  './datasets/retweet_norm/test.json',
                'data_specs': {
                    'num_event_types':    NUM_EVENT_TYPES,
                    'pad_token_id':       NUM_EVENT_TYPES,  # pad = num_types
                    'padding_side':       'right',
                    'truncation_side':    'right',
                    'truncation_strategy': 'longest_first',
                    'max_len':            max_len,
                },
            },
        },
        exp_id: {
            'base_config': {
                'stage':      stage,
                'backend':    'torch',
                'dataset_id': DATASET_NAME,
                'runner_id':  'std_tpp',
                'model_id':   model_id,
                'base_dir':   f'./checkpoints/retweet_norm/{model_id}/seed{seed}/',
            },
            'trainer_config': trainer_cfg,
            'model_config':  model_cfg,
        },
    }
    return config, exp_id


def write_yaml_and_load(config_dict, experiment_id):
    """Escreve YAML em arquivo temporario e retorna Config."""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        yaml.dump(config_dict, f, default_flow_style=False)
        tmp_path = f.name
    try:
        cfg = Config.build_from_yaml_file(tmp_path, experiment_id=experiment_id)
    finally:
        os.unlink(tmp_path)
    return cfg


def get_model_dir(model_id, seed):
    """Retorna o caminho real do checkpoint salvo pelo EasyTPP."""
    base = f'./checkpoints/retweet_norm/{model_id}/seed{seed}'
    if not os.path.isdir(base):
        return f'{base}/models/saved_model'
    candidates = []
    for entry in os.scandir(base):
        if entry.is_dir():
            candidate = os.path.join(entry.path, 'models', 'saved_model')
            if os.path.exists(candidate):
                candidates.append((entry.stat().st_mtime, candidate))
    if candidates:
        return sorted(candidates)[-1][1]
    return f'{base}/models/saved_model'


print('Funcoes auxiliares definidas.', flush=True)

In [ ]:
# ── Treinamento com captura de NLL por epoca ─────────────────────────
# Treina cada (modelo, seed) e captura as perdas por epoca via redirect_stdout.
# Progresso salvo apos cada run para sobreviver a quedas de conexao.

import logging

progress = load_progress()

total_runs = len(MODELS) * len(SEEDS)
done_runs = len(progress.get('trained', set()))
print(f'Treinamentos: {done_runs}/{total_runs} ja concluidos\n', flush=True)

for model_id in MODELS:
    for seed in SEEDS:
        key = f"{model_id}_seed{seed}"

        # Verifica se ja treinou E o checkpoint ainda existe
        if key in progress.get('trained', set()):
            model_dir = get_model_dir(model_id, seed)
            if os.path.exists(model_dir):
                t = progress['train_times'].get(key)
                t_str = f'{t:.0f}s ({t/60:.1f}min)' if t else 'N/A'
                print(f'  [SKIP] {key}  (treino: {t_str})', flush=True)
                continue
            else:
                print(f'  [RETRAIN] {key} — checkpoint perdido, re-treinando...', flush=True)
                progress['trained'].discard(key)
                progress['train_times'].pop(key, None)
                progress['epoch_logs'].pop(key, None)
                progress['results'] = [
                    r for r in progress['results']
                    if not (r['model'] == model_id and r['seed'] == seed)
                ]
                save_progress(progress)

        print(f'\n{"="*60}', flush=True)
        print(f'  {model_id} / seed={seed}  [max_len={TRAIN_MAX_LEN}]', flush=True)
        print(f'{"="*60}', flush=True)

        cfg_dict, exp_id = make_yaml(model_id, seed, TRAIN_MAX_LEN, stage='train')
        cfg = write_yaml_and_load(cfg_dict, experiment_id=exp_id)
        runner = Runner.build_from_config(cfg)

        # Captura stdout para extrair NLL por epoca
        buf = io.StringIO()
        t0 = time.time()

        # Redireciona tanto stdout quanto logging para capturar epoch losses
        root_logger = logging.getLogger()
        old_handlers = root_logger.handlers[:]
        stream_handler = logging.StreamHandler(buf)
        stream_handler.setLevel(logging.DEBUG)
        root_logger.addHandler(stream_handler)

        try:
            with contextlib.redirect_stdout(buf):
                runner.run()
        finally:
            root_logger.removeHandler(stream_handler)

        elapsed = time.time() - t0

        # Parse epoch losses do output capturado
        output = buf.getvalue()

        # Mostra as ultimas linhas do output para debug
        lines = output.strip().split('\n')
        print(f'\n--- Ultimas 10 linhas do output ---', flush=True)
        for line in lines[-10:]:
            print(f'  {line}', flush=True)

        # Tenta extrair epoch losses com varios padroes
        epoch_losses = []
        for line in lines:
            # Padrao 1: "Epoch 5/100 - loss: 1.234"
            match = re.search(r'[Ee]poch\s*[\[:]?\s*(\d+).*?loss[:\s=]*([0-9.e+-]+)', line)
            if match:
                epoch_losses.append((int(match.group(1)), float(match.group(2))))
                continue
            # Padrao 2: "train_loss" ou "nll" em logs
            match = re.search(r'(?:train_loss|nll|loglike)[:\s=]*([0-9.e+-]+)', line, re.IGNORECASE)
            if match:
                epoch_losses.append((len(epoch_losses) + 1, float(match.group(1))))

        del runner
        free_gpu()

        progress['trained'].add(key)
        progress['epoch_logs'][key] = epoch_losses
        progress['train_times'][key] = elapsed
        save_progress(progress)
        print(f'\n  [OK] {key}: {elapsed:.0f}s ({elapsed/60:.1f}min), '
              f'{len(epoch_losses)} epoch logs capturados', flush=True)
        print(f'  Checkpoint: {get_model_dir(model_id, seed)}', flush=True)

print(f'\nTreinamento concluido! {len(progress["trained"])}/{total_runs}', flush=True)

In [ ]:
# ── Avaliacao: 1x (truncado) e 5x (extrapolacao) ─────────────────────

progress = load_progress()

done_evals = set()
for r in progress['results']:
    done_evals.add((r['model'], r['seed'], r['extrap']))

total_evals = len(MODELS) * len(SEEDS) * 2  # 1x + 5x
print(f'Avaliacoes: {len(done_evals)}/{total_evals} ja concluidas\n', flush=True)

for model_id in MODELS:
    for seed in SEEDS:
        model_dir = get_model_dir(model_id, seed)
        if not os.path.exists(model_dir):
            print(f'  [WARN] Checkpoint nao encontrado: {model_dir}', flush=True)
            continue

        for extrap_name, max_len in [('1x', EVAL_1X), ('5x', EVAL_5X)]:
            eval_key = (model_id, seed, extrap_name)
            if eval_key in done_evals:
                print(f'  [SKIP] {model_id}/seed{seed}/{extrap_name}', flush=True)
                continue

            print(f'  Eval: {model_id}/seed{seed}'
                  f' [{extrap_name}, max_len={max_len}]', end=' -> ', flush=True)

            cfg_dict, exp_id = make_yaml(
                model_id, seed, max_len,
                stage='eval', pretrained_model_dir=model_dir)
            cfg = write_yaml_and_load(cfg_dict, experiment_id=exp_id)
            runner = Runner.build_from_config(cfg)

            test_loader = runner._data_loader.test_loader()
            metrics = runner._evaluate_model(test_loader)
            del runner
            free_gpu()

            result = {
                'dataset':  DATASET_NAME,
                'model':    model_id,
                'seed':     seed,
                'extrap':   extrap_name,
                'max_len':  max_len,
                'nll':      -metrics.get('loglike', float('nan')),
                'acc':      metrics.get('acc', float('nan')),
                'rmse':     metrics.get('rmse', float('nan')),
            }
            progress['results'].append(result)
            save_progress(progress)
            print(f'NLL={result["nll"]:.4f}  ACC={result["acc"]:.4f}  '
                  f'RMSE={result["rmse"]:.4f}', flush=True)

print(f'\nAvaliacao concluida! {len(progress["results"])} resultados salvos.', flush=True)

In [ ]:
# ── Tabela de resultados ──────────────────────────────────────────────

progress = load_progress()
df = pd.DataFrame(progress['results'])

if len(df) == 0:
    print('Nenhum resultado disponivel. Execute as celulas de treino e avaliacao primeiro.', flush=True)
else:
    print('=' * 90, flush=True)
    print('RESULTADOS — ABLACAO RETWEET NORMALIZADO', flush=True)
    print('=' * 90, flush=True)

    # Tabela detalhada por seed
    print(f'\n{"Model":<10} {"Seed":>6} {"NLL_1x":>10} {"NLL_5x":>10} '
          f'{"dNLL":>10} {"Acc_1x":>10} {"Acc_5x":>10} {"RMSE_1x":>10} {"RMSE_5x":>10}', flush=True)
    print('-' * 90, flush=True)

    for model_id in MODELS:
        for seed in SEEDS:
            sub = df[(df['model'] == model_id) & (df['seed'] == seed)]
            nll_1x = sub[sub['extrap'] == '1x']['nll'].values
            nll_5x = sub[sub['extrap'] == '5x']['nll'].values
            acc_1x = sub[sub['extrap'] == '1x']['acc'].values
            acc_5x = sub[sub['extrap'] == '5x']['acc'].values
            rmse_1x = sub[sub['extrap'] == '1x']['rmse'].values
            rmse_5x = sub[sub['extrap'] == '5x']['rmse'].values

            n1 = f'{nll_1x[0]:.4f}' if len(nll_1x) > 0 else '  ---'
            n5 = f'{nll_5x[0]:.4f}' if len(nll_5x) > 0 else '  ---'
            dn = f'{nll_5x[0] - nll_1x[0]:.4f}' if len(nll_1x) > 0 and len(nll_5x) > 0 else '  ---'
            a1 = f'{acc_1x[0]:.4f}' if len(acc_1x) > 0 else '  ---'
            a5 = f'{acc_5x[0]:.4f}' if len(acc_5x) > 0 else '  ---'
            r1 = f'{rmse_1x[0]:.4f}' if len(rmse_1x) > 0 else '  ---'
            r5 = f'{rmse_5x[0]:.4f}' if len(rmse_5x) > 0 else '  ---'

            print(f'{model_id:<10} {seed:>6} {n1:>10} {n5:>10} {dn:>10} '
                  f'{a1:>10} {a5:>10} {r1:>10} {r5:>10}', flush=True)

    # Tabela resumo (media +/- std)
    print(f'\n{"="*90}', flush=True)
    print('MEDIA +/- STD (3 seeds)', flush=True)
    print(f'{"="*90}', flush=True)
    print(f'{"Model":<10} {"NLL_1x":>18} {"NLL_5x":>18} '
          f'{"Acc_1x":>18} {"Acc_5x":>18}', flush=True)
    print('-' * 90, flush=True)

    for model_id in MODELS:
        for extrap in ['1x', '5x']:
            sub = df[(df['model'] == model_id) & (df['extrap'] == extrap)]
            if len(sub) == 0:
                continue

        sub_1x = df[(df['model'] == model_id) & (df['extrap'] == '1x')]
        sub_5x = df[(df['model'] == model_id) & (df['extrap'] == '5x')]

        n1 = f'{sub_1x["nll"].mean():.4f}+/-{sub_1x["nll"].std():.4f}' if len(sub_1x) > 0 else '---'
        n5 = f'{sub_5x["nll"].mean():.4f}+/-{sub_5x["nll"].std():.4f}' if len(sub_5x) > 0 else '---'
        a1 = f'{sub_1x["acc"].mean():.4f}+/-{sub_1x["acc"].std():.4f}' if len(sub_1x) > 0 else '---'
        a5 = f'{sub_5x["acc"].mean():.4f}+/-{sub_5x["acc"].std():.4f}' if len(sub_5x) > 0 else '---'

        print(f'{model_id:<10} {n1:>18} {n5:>18} {a1:>18} {a5:>18}', flush=True)

In [ ]:
# ── Grafico: Convergencia do treino (NLL por epoca) ──────────────────

progress = load_progress()

COLORS = {'RoTHP': '#2196F3', 'HoTHP': '#4CAF50'}

fig, ax = plt.subplots(figsize=(10, 5))

has_data = False
for model_id in MODELS:
    all_losses = []
    for seed in SEEDS:
        key = f"{model_id}_seed{seed}"
        losses = progress['epoch_logs'].get(key, [])
        if losses:
            all_losses.append([l[1] for l in losses])

    if all_losses:
        has_data = True
        min_len = min(len(l) for l in all_losses)
        aligned = np.array([l[:min_len] for l in all_losses])
        mean_loss = aligned.mean(axis=0)
        std_loss = aligned.std(axis=0)
        epochs = np.arange(1, min_len + 1)

        color = COLORS[model_id]
        ax.plot(epochs, mean_loss, color=color, label=model_id, linewidth=2)
        ax.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss,
                        color=color, alpha=0.2)

if has_data:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Training NLL')
    ax.set_title('Training Convergence — Retweet Normalizado\n'
                 'RoTHP vs HoTHP (media +/- std, 3 seeds)')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('retweet_norm_convergence.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvo: retweet_norm_convergence.png', flush=True)
else:
    print('Sem dados de epoch logs. Verifique se o treino capturou as perdas.', flush=True)
    print('(Os epoch logs dependem do formato de logging do EasyTPP — pode ser necessario', flush=True)
    print(' ajustar os padroes regex na celula de treinamento.)', flush=True)

In [ ]:
# ── Grafico: Comparacao com benchmark original (sem normalizacao) ─────
# Preencha os valores do benchmark original abaixo (de Benchmark_Real_Datasets_Colab)

progress = load_progress()
df = pd.DataFrame(progress['results'])

# ===== PREENCHER COM RESULTADOS DO BENCHMARK ORIGINAL =====
# Formato: (mean_nll, std_nll) do retweet original, avaliado em 5x (max_len=250)
ORIGINAL_RESULTS = {
    'RoTHP': {'nll_5x': (None, None), 'nll_1x': (None, None)},  # (mean, std)
    'HoTHP': {'nll_5x': (None, None), 'nll_1x': (None, None)},  # (mean, std)
}
# ===========================================================

# Coleta resultados normalizados
norm_results = {}
for model_id in MODELS:
    for extrap in ['1x', '5x']:
        sub = df[(df['model'] == model_id) & (df['extrap'] == extrap)]
        if len(sub) > 0:
            norm_results[f'{model_id}_nll_{extrap}'] = (sub['nll'].mean(), sub['nll'].std())

# Verifica se temos dados para o grafico
has_original = any(v['nll_5x'][0] is not None for v in ORIGINAL_RESULTS.values())
has_norm = len(norm_results) > 0

if has_norm:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax_idx, extrap in enumerate(['1x', '5x']):
        ax = axes[ax_idx]
        x = np.arange(len(MODELS))
        width = 0.35

        # Barras: original
        orig_means = []
        orig_stds = []
        for model_id in MODELS:
            m, s = ORIGINAL_RESULTS.get(model_id, {}).get(f'nll_{extrap}', (None, None))
            orig_means.append(m if m is not None else 0)
            orig_stds.append(s if s is not None else 0)

        # Barras: normalizado
        norm_means = []
        norm_stds = []
        for model_id in MODELS:
            key = f'{model_id}_nll_{extrap}'
            if key in norm_results:
                norm_means.append(norm_results[key][0])
                norm_stds.append(norm_results[key][1])
            else:
                norm_means.append(0)
                norm_stds.append(0)

        if has_original:
            bars1 = ax.bar(x - width/2, orig_means, width, yerr=orig_stds,
                          capsize=4, label='Original (raw)', color='#E57373', alpha=0.85)
        bars2 = ax.bar(x + width/2 if has_original else x, norm_means, width,
                      yerr=norm_stds, capsize=4, label='Normalizado (dt/mean)',
                      color='#64B5F6', alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels(MODELS, fontsize=11)
        ax.set_ylabel('NLL (menor = melhor)')
        ax.set_title(f'Retweet — {extrap}\n(max_len={EVAL_1X if extrap == "1x" else EVAL_5X})',
                     fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, axis='y')

    fig.suptitle('Ablacao: Efeito da Normalizacao Temporal no Retweet\n'
                 'RoTHP vs HoTHP (3 seeds)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('retweet_norm_vs_original.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvo: retweet_norm_vs_original.png', flush=True)

    if not has_original:
        print('\nNOTA: Valores do benchmark original nao preenchidos.', flush=True)
        print('Preencha ORIGINAL_RESULTS com os valores de Benchmark_Real_Datasets_Colab', flush=True)
        print('e re-execute esta celula para ver a comparacao completa.', flush=True)
else:
    print('Sem resultados normalizados. Execute as celulas de avaliacao primeiro.', flush=True)

## Conclusoes

**Preencher apos execucao:**

1. **Normalizacao corrigiu o HoTHP?**
   - NLL_5x HoTHP normalizado vs original: ___
   - Se melhorou significativamente, confirma hipotese de overflow numerico

2. **RoTHP foi afetado pela normalizacao?**
   - RoTHP e robusto a escala temporal (rotary embeddings operam em fase relativa)
   - Espera-se pouca variacao

3. **Ranking apos normalizacao:**
   - Se HoTHP ~= RoTHP: o problema era puramente numerico
   - Se HoTHP < RoTHP: normalizacao ajudou mas nao e suficiente
   - Se HoTHP > RoTHP: HoTHP e superior quando a escala temporal e controlada

4. **Implicacoes para o design do modelo:**
   - Se normalizacao resolve: considerar layer de normalizacao temporal interna ao HoTHP
   - Se nao resolve: problema e arquitetural, nao numerico